In [ ]:
# Install playwright, its system dependencies, and the chromium browser
!pip install playwright
!playwright install-deps chromium
!playwright install chromium

In [2]:
import asyncio
from playwright.async_api import async_playwright

async def extract_blog_links(start_page=1, end_page=481):
    all_unique_links = set()

    async with async_playwright() as p:
        # Launch browser
        browser = await p.chromium.launch(headless=True)
        # Set a standard desktop viewport
        page = await browser.new_page(viewport={'width': 1280, 'height': 800})

        # Filter console logs
        page.on("console", lambda msg: print(f"Browser: {msg.text}") if msg.type == "log" else None)

        print(f"Starting to scrape blog links from page {start_page} to {end_page}...")

        # Loop through all the pages
        for current_page in range(start_page, end_page + 1):
            url = f"https://www.nawy.com/blog/category/areas/page/{current_page}"

            try:
                # domcontentloaded is usually faster than networkidle and sufficient for standard paginated blogs
                await page.goto(url, timeout=60000, wait_until="domcontentloaded")

                target_selector = "h3.entry-title.td-module-title a"

                try:
                    # Wait for the blog link elements to appear
                    await page.wait_for_selector(target_selector, timeout=15000)
                except Exception:
                    print(f"Could not find any links on page {current_page}. Stopping the scrape early.")
                    # If we can't find the selector, we've likely hit the end of the blog pages
                    break

                # Evaluate JavaScript on the page to extract all hrefs matching the selector
                page_links = await page.evaluate("""
                    (selector) => {
                        const links = Array.from(document.querySelectorAll(selector));
                        return links.map(a => a.href);
                    }
                """, target_selector)

                # Track how many new ones we found to display progress
                before_count = len(all_unique_links)
                all_unique_links.update(page_links)
                new_links_count = len(all_unique_links) - before_count

                print(f"Page {current_page}: Found {len(page_links)} links ({new_links_count} new). Total unique: {len(all_unique_links)}")

            except Exception as e:
                print(f"Error navigating to page {current_page}: {e}")
                # Continue to the next page instead of completely crashing if one page times out
                continue

        await browser.close()
        # Convert set back to a list
        return list(all_unique_links)

async def main():
    # Set the range based on your requirement
    links = await extract_blog_links(start_page=1, end_page=481)

    if links:
        output_file = 'blog_links.txt'
        # Sort links alphabetically for a clean text file
        links.sort()

        with open(output_file, 'w', encoding='utf-8') as f:
            for link in links:
                f.write(f"{link}\n")

        print(f"\nSuccess! Total unique blog links extracted: {len(links)}")
        print(f"Full list of links saved to: {output_file}")
    else:
        print("No links were extracted.")

if __name__ == "__main__":
    try:
        asyncio.run(main())
    except RuntimeError:
        # Fallback for Jupyter/Colab event loop rules
        import nest_asyncio
        nest_asyncio.apply()
        asyncio.get_event_loop().run_until_complete(main())

Starting to scrape blog links from page 1 to 481...
Browser: JQMIGRATE: Migrate is installed, version 3.4.1
Page 1: Found 14 links (10 new). Total unique: 10
Browser: JQMIGRATE: Migrate is installed, version 3.4.1
Page 2: Found 14 links (5 new). Total unique: 15
Browser: JQMIGRATE: Migrate is installed, version 3.4.1
Page 3: Found 14 links (5 new). Total unique: 20
Browser: JQMIGRATE: Migrate is installed, version 3.4.1
Page 4: Found 14 links (5 new). Total unique: 25
Browser: JQMIGRATE: Migrate is installed, version 3.4.1
Page 5: Found 14 links (5 new). Total unique: 30
Browser: JQMIGRATE: Migrate is installed, version 3.4.1
Page 6: Found 14 links (5 new). Total unique: 35
Browser: JQMIGRATE: Migrate is installed, version 3.4.1
Page 7: Found 14 links (5 new). Total unique: 40
Browser: JQMIGRATE: Migrate is installed, version 3.4.1
Page 8: Found 14 links (5 new). Total unique: 45
Browser: JQMIGRATE: Migrate is installed, version 3.4.1
Page 9: Found 14 links (5 new). Total unique: 50
Br

/tmp/ipykernel_14530/50221353.py:85: RuntimeWarning: coroutine 'main' was never awaited
  asyncio.get_event_loop().run_until_complete(main())
